# 11. Bias in inference and mitigation: loan approval

The loan table from [WorkshopIgualdad2025](https://github.com/rferper/WorkshopIgualdad2025) is the one the workshop used to **mitigate** gender bias: drop the sensitive attribute, reweigh samples (Kamiran and Calders 2012), or penalise demographic parity in the genetic loss.

Those three ideas are now `ebdai.reweigh_weights`, `ebdai.weighted_mcc_loss` and `ebdai.fairness_regularized_loss`, and they plug into `ex_fuzzy.BaseFuzzyRulesClassifier.customized_loss`.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

from ex_fuzzy import BaseFuzzyRulesClassifier, FUZZY_SETS, eval_tools
from ebdai import (
    features_and_target, load_loan_approval, outcome_rates_by_group,
    plot_outcome_rates, fairness_report, reweigh_weights,
    weighted_mcc_loss, fairness_regularized_loss,
    parse_printed_rules, winning_rules_by_group,
    plot_winning_rules_by_group,
)

frame, sensitive = load_loan_approval()
X, y = features_and_target(frame, 'Loan_Status')
print(X.head())
rates = outcome_rates_by_group(y, X[sensitive], positive_label=1)
print(rates)
plot_outcome_rates(rates, title='Loan approval rate by gender')

  Gender Married Dependents     Education Self_Employed  ApplicantIncome  \
1   Male     Yes          1      Graduate            No             4583   
2   Male     Yes          0      Graduate           Yes             3000   
3   Male     Yes          0  Not Graduate            No             2583   
4   Male      No          0      Graduate            No             6000   
5   Male     Yes          2      Graduate           Yes             5417   

   CoapplicantIncome  LoanAmount  Loan_Amount_Term  Credit_History  \
1             1508.0       128.0             360.0             1.0   
2                0.0        66.0             360.0             1.0   
3             2358.0       120.0             360.0             1.0   
4                0.0       141.0             360.0             1.0   
5             4196.0       267.0             360.0             1.0   

  Property_Area  
1         Rural  
2         Urban  
3         Urban  
4         Urban  
5         Urban  
    group    n

<Axes: title={'center': 'Loan approval rate by gender'}, xlabel='Group', ylabel='Positive rate'>

## Baseline fuzzy rules (sensitive attribute kept)

In [2]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42
)

def fit_rules(X_tr, y_tr, X_te, y_te, loss=None):
    clf = BaseFuzzyRulesClassifier(
        nRules=8, nAnts=3, fuzzy_type=FUZZY_SETS.t1,
        n_linguistic_variables=3, ds_mode=2, verbose=False,
        n_gen=6, pop_size=12, patience=3, random_state=42,
    )
    if loss is not None:
        clf.customized_loss(loss)
    clf.fit(X_tr, y_tr)
    report = eval_tools.eval_fuzzy_model(
        clf, X_tr, y_tr, X_te, y_te,
        plot_rules=False, print_rules=False, plot_partitions=False,
        return_rules=True, bootstrap_results_print=False,
    )
    return clf, report

clf, report = fit_rules(X_train, y_train, X_test, y_test)
y_pred = clf.predict(X_test)
table, gaps = fairness_report(y_test, y_pred, X_test[sensitive])
print('baseline')
print(table)
print(pd.Series(gaps))

------------
ACCURACY
Train performance: 0.5669781931464174
Test performance: 0.5911949685534591
------------
MATTHEW CORRCOEF
Train performance: 0.10374834639844925
Test performance: 0.16926832390925828
------------
baseline
    group    n  selection_rate       tpr       fpr       fnr       tnr
0    Male  137        0.576642  0.628866  0.450000  0.371134  0.550000
1  Female   22        0.272727  0.333333  0.142857  0.666667  0.857143
demographic_parity_difference    0.303915
demographic_parity_ratio         0.472957
equalized_odds_difference        0.307143
equalized_odds_ratio             0.317460
tpr_difference                   0.295533
fpr_difference                   0.307143
dtype: float64


## Reweighing

Kamiran and Calders weights make P(Y, A) independent in the training sample. Pass them through `weighted_mcc_loss` into `customized_loss`.

In [3]:
weights = reweigh_weights(y_train, X_train[sensitive])
clf_w, _ = fit_rules(
    X_train, y_train, X_test, y_test,
    loss=weighted_mcc_loss(weights),
)
table_w, gaps_w = fairness_report(
    y_test, clf_w.predict(X_test), X_test[sensitive]
)
print('reweighed')
print(table_w)
print(pd.Series(gaps_w))

------------
ACCURACY
Train performance: 0.49221183800623053
Test performance: 0.5220125786163522
------------
MATTHEW CORRCOEF
Train performance: 0.10050895214863548
Test performance: 0.15053717375108475
------------
reweighed
    group    n  selection_rate       tpr       fpr       fnr       tnr
0    Male  137        0.328467  0.381443  0.200000  0.618557  0.800000
1  Female   22        0.772727  0.800000  0.714286  0.200000  0.285714
demographic_parity_difference    0.444260
demographic_parity_ratio         0.425075
equalized_odds_difference        0.514286
equalized_odds_ratio             0.280000
tpr_difference                   0.418557
fpr_difference                   0.514286
dtype: float64


## Fairness regularisation

The genetic objective becomes MCC minus λ times the demographic-parity difference on the training rows (λ = 0.2, as in the workshop).

In [4]:
clf_f, report_f = fit_rules(
    X_train, y_train, X_test, y_test,
    loss=fairness_regularized_loss(X_train[sensitive], lam=0.2),
)
table_f, gaps_f = fairness_report(
    y_test, clf_f.predict(X_test), X_test[sensitive]
)
print('regularised')
print(table_f)
print(pd.Series(gaps_f))
counts = winning_rules_by_group(
    clf_f, X_test, X_test[sensitive],
    rule_texts=parse_printed_rules(report_f or ''),
)
plot_winning_rules_by_group(
    counts, title='Winning loan rules by gender (regularised fit)'
)

------------
ACCURACY
Train performance: 0.4174454828660436
Test performance: 0.4276729559748428
------------
MATTHEW CORRCOEF
Train performance: 0.09636798445458983
Test performance: 0.09580279131572642
------------
regularised
    group    n  selection_rate       tpr       fpr       fnr       tnr
0    Male  137        0.240876  0.268041  0.175000  0.731959  0.825000
1  Female   22        0.181818  0.200000  0.142857  0.800000  0.857143
demographic_parity_difference    0.059058
demographic_parity_ratio         0.754821
equalized_odds_difference        0.068041
equalized_odds_ratio             0.746154
tpr_difference                   0.068041
fpr_difference                   0.032143
dtype: float64


<Axes: title={'center': 'Winning loan rules by gender (regularised fit)'}, xlabel='Winning rule', ylabel='Share of group'>